# Module 2 · Lesson 03: Chain-of-Thought Prompting

**Chain-of-Thought (CoT)** prompting asks the model to "think step by step".
This dramatically improves accuracy on reasoning, math, and logic tasks.

## What you will learn
1. Why CoT works (and when it doesn't)
2. Zero-shot CoT ("think step by step")
3. Few-shot CoT (reasoning examples)
4. CoT for **code debugging**
5. CoT for **decision making**
6. Different **trigger phrases**

In [1]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / "module_02_prompt_enginnering/.env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
OPENAI_MODEL = "gpt-4o-mini" 
if client:
    print(f"OpenAI client ready - using model {OPENAI_MODEL}")

OpenAI client ready - using model gpt-4o-mini


In [2]:
def ask_open_ai(prompt: str, system_content: str= None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model:  str="gpt-4o-mini") -> str:
    """
    Send a prompt to the OpenAI Chat Completions API and return the assistant's reply.

    This helper function builds a chat message list, optionally including a system
    instruction, sends it to the specified model, and returns the generated text
    content from the first response choice.

    Args:
        prompt (str): The user prompt to send to the model.
        system_content (str, optional): Optional system-level instruction that
            defines the assistant's behavior, tone, or constraints. If provided,
            it is added as the first message in the conversation. Defaults to None.
        temperature (float, optional): Sampling temperature used to control
            randomness in the model's response. Lower values produce more
            deterministic outputs; higher values produce more creative or varied
            outputs. Defaults to 0.7.
        max_resp_tokens (int, optional): Maximum number of tokens allowed in the
            generated response. Defaults to 800.
        ai_model (str, optional): The model name to use for the request.
            Defaults to "gpt-4o-mini".

    Returns:
        str: The text content of the assistant's first response message.

    Raises:
        Exception: Propagates any exception raised by the OpenAI client request
            (for example, authentication errors, rate limits, invalid parameters,
            or network-related failures).

    Example of use:
         reply = ask("Explain recursion in simple terms.")
         print(reply)

        reply = ask(
             prompt="Write a haiku about Python.",
             system_content="You are a poetic assistant.",
             temperature=0.9,
             max_resp_tokens=100,
             ai_model="gpt-4o-mini"
        )        
        print(reply)
    """
    msgs = []
    if system_content:
        msgs.append({"role": "system", "content": system_content})
    msgs.append({"role":"user", "content": prompt})
    resp = client.chat.completions.create(
        model=ai_model,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_resp_tokens
    )
    return resp.choices[0].message.content

display(Markdown(f"**The function has run**"))

**The function has run**

---
## 1. The Problem: Without CoT

Let's see what happens when we ask a tricky math problem *without* CoT:

In [3]:
prompt = """A store has 15 apples. It sells 7 in the morning and receives a delivery 12.
Then it sells 5 more in the afternoon. How many apples remain?
"""

direct_answer = ask_open_ai(prompt=f"{prompt}\n\nAnswer just the number.")
print(f"Direct answer (no CoT): {direct_answer}")

Direct answer (no CoT): 15


---
## 2. Zero-Shot CoT — "Let's think step by step"

Simply adding **"Let's think step by step"** triggers reasoning:

In [4]:
cot_answer = ask_open_ai(prompt= f"{prompt}\n\nLet's think step by step. Use plain text, not Latex.")

display(Markdown(f"### Step-by-step Reasoning\n\n{cot_answer.strip()}"))

### Step-by-step Reasoning

Sure! Let's break it down step by step.

1. The store starts with 15 apples.
2. In the morning, it sells 7 apples. 
   - So, 15 - 7 = 8 apples remaining.
3. Then, the store receives a delivery of 12 apples.
   - Now, 8 + 12 = 20 apples.
4. In the afternoon, it sells 5 more apples.
   - So, 20 - 5 = 15 apples remaining.

Therefore, the store has 15 apples left.

In [5]:
problem = '''John has 3 boxes.
Each box contains 5 bags.
Each bag contains 4 marbles.

He gives away 18 marbles to his friend.

Then he buys 2 more boxes with the same structure.

How many marbles does he have now?
'''

In [6]:
derect_answer = ask_open_ai(prompt=f"{problem}\n\n Answer with just the number.",  temperature=0)
print(direct_answer)

15


In [7]:
cot_answer = ask_open_ai(prompt= f"{problem}\n\nLet's think step by step. Use plain text, not Latex.")

display(Markdown(f"### Step-by-step Reasoning\n\n{cot_answer.strip()}"))

### Step-by-step Reasoning

Let's break the problem down step by step.

1. **Calculate the initial number of marbles:**
   - John has 3 boxes.
   - Each box contains 5 bags.
   - Each bag contains 4 marbles.

   To find the total number of marbles:
   - First, calculate the total number of bags: 
     3 boxes * 5 bags/box = 15 bags.
   - Next, calculate the total number of marbles:
     15 bags * 4 marbles/bag = 60 marbles.

2. **After giving away marbles:**
   - John gives away 18 marbles.
   - Therefore, the number of marbles he has left:
     60 marbles - 18 marbles = 42 marbles.

3. **After buying more boxes:**
   - John buys 2 more boxes.
   - Each of these boxes also has 5 bags and each bag contains 4 marbles.

   To calculate the number of marbles in the new boxes:
   - First, calculate the total number of bags in the 2 new boxes:
     2 boxes * 5 bags/box = 10 bags.
   - Next, calculate the total number of marbles in these new bags:
     10 bags * 4 marbles/bag = 40 marbles.

4. **Calculate the total number of marbles John has now:**
   - Add the marbles he has left after giving some away to the marbles he got from the new boxes:
     42 marbles + 40 marbles = 82 marbles.

So, John has a total of 82 marbles now.

> 💡 **The research:** Kojima et al. (2022) showed that adding "Let's think step by step" 
> improved accuracy by 10–40% on reasoning benchmarks — with zero examples!

---
## 3. Logical Reasoning

In [9]:
logic = '''Consider these statements:
1. All programmers drink coffee.
2. Some coffee drinkers are "night owls".
3. No night owls wake up early.

Question: Can we conclude then some programmers don't wake up early?
Think thought step by step, examining each logic connection.
'''
result = ask_open_ai(logic)
display(Markdown(result))

Let's analyze the statements step by step.

1. **Statement 1: All programmers drink coffee.**
   - This means that every individual who is a programmer is also someone who drinks coffee. We can represent this as:
     - If \( P \) represents a programmer, then \( P \Rightarrow C \) (where \( C \) represents a coffee drinker).

2. **Statement 2: Some coffee drinkers are "night owls".**
   - This means that there is at least one individual who drinks coffee and is also a night owl. We can represent this as:
     - There exists an individual \( x \) such that \( C(x) \Rightarrow N(x) \) (where \( N \) represents a night owl).

3. **Statement 3: No night owls wake up early.**
   - This means that if someone is a night owl, they do not wake up early. We can represent this as:
     - If \( N \) then not \( E \) (where \( E \) represents waking up early).

Now, let's combine these statements step by step:

- From **Statement 1**, we know that all programmers are coffee drinkers. Therefore, if we take a programmer, they must also be a coffee drinker.
  
- From **Statement 2**, since some coffee drinkers are night owls, we can conclude that there exists at least one coffee drinker who is a night owl. Since all programmers are coffee drinkers, it is possible that this night owl could be a programmer. 

- From **Statement 3**, we know that night owls do not wake up early. Therefore, if we consider the coffee drinker who is also a night owl (from **Statement 2**), we can conclude that they do not wake up early.

Now, since we have established that at least some coffee drinkers are night owls and that all programmers are coffee drinkers, there is a possibility that some of those coffee drinkers who are night owls are indeed programmers. 

Thus, we can conclude that at least some programmers (those who are also coffee drinkers and night owls) do not wake up early.

### Conclusion:
Yes, we can conclude that some programmers don't wake up early.

---
## 4. CoT for Code Debugging

CoT is particularly effective for finding bugs — it forces the model to trace execution:

In [10]:
# ── Code debugging with CoT ──────────────────────────
debug_prompt = """Find the bug in this Python code:
 
```python
def find_duplicates(lst):
    seen = set()
    duplicates = set()
    for item in lst:
        if item in seen:
            duplicates.add(item)
    seen.add(item)
    return list(duplicates)
 
# Expected: [2, 3]
# Actual:   []
print(find_duplicates([1, 2, 2, 3, 3, 4]))
```
 
Trace through the code step by step with the input [1, 2, 2, 3, 3, 4]."""
 
result = ask_open_ai(debug_prompt)
display(Markdown(result))

Let's trace through the provided `find_duplicates` function step by step with the input list `[1, 2, 2, 3, 3, 4]`.

1. **Initialization**: 
   - `seen` is initialized as an empty set: `seen = set()`.
   - `duplicates` is initialized as an empty set: `duplicates = set()`.

2. **Iteration over the list**:
   - The function will loop over each `item` in the input list `lst`.

3. **First iteration** (`item = 1`):
   - Check if `item` (1) is in `seen`: `1 in seen` → `False`.
   - Since it's not in `seen`, we do not add it to `duplicates`.
   - We add `1` to `seen`: `seen = {1}`.

4. **Second iteration** (`item = 2`):
   - Check if `item` (2) is in `seen`: `2 in seen` → `False`.
   - Since it's not in `seen`, we do not add it to `duplicates`.
   - We add `2` to `seen`: `seen = {1, 2}`.

5. **Third iteration** (`item = 2`):
   - Check if `item` (2) is in `seen`: `2 in seen` → `True`.
   - Since it is in `seen`, we add `2` to `duplicates`: `duplicates = {2}`.
   - We then attempt to add `item` (2) to `seen`, but `2` is already there, so `seen` remains `{1, 2}`.

6. **Fourth iteration** (`item = 3`):
   - Check if `item` (3) is in `seen`: `3 in seen` → `False`.
   - Since it's not in `seen`, we do not add it to `duplicates`.
   - We add `3` to `seen`: `seen = {1, 2, 3}`.

7. **Fifth iteration** (`item = 3`):
   - Check if `item` (3) is in `seen`: `3 in seen` → `True`.
   - Since it is in `seen`, we add `3` to `duplicates`: `duplicates = {2, 3}`.
   - We then attempt to add `item` (3) to `seen`, but `3` is already there, so `seen` remains `{1, 2, 3}`.

8. **Sixth iteration** (`item = 4`):
   - Check if `item` (4) is in `seen`: `4 in seen` → `False`.
   - Since it's not in `seen`, we do not add it to `duplicates`.
   - We add `4` to `seen`: `seen = {1, 2, 3, 4}`.

9. **End of loop**: The loop ends after processing all items in `lst`.

10. **Return statement**: The function returns the list version of `duplicates`: `return list(duplicates)`.

### Key Issue
The bug in the code is that the line `seen.add(item)` is outside the loop. This means that `item` is only added to `seen` after the loop has finished, and thus the first occurrence of each item is never recorded in `seen` when it should be.

### Corrected Code
To fix the code, we should move the line `seen.add(item)` inside the loop:

```python
def find_duplicates(lst):
    seen = set()
    duplicates = set()
    for item in lst:
        if item in seen:
            duplicates.add(item)
        seen.add(item)  # Move this line inside the loop


---
## 5. Few-Shot CoT

Providing **reasoning examples** teaches the model HOW to think, not just what to output:

In [11]:
prompt = '''Solve word problems step by step. Use plain text, not LaTeX.

Q: If John has 3 boxes with 5 apples each, and he gives away 4 apples, how many does he have?
A: Step 1: John starts with 3 x 5 = 15 apples.
   Step 2: He gives away 4, so 15 - 4 = 11 apples.
   Answer: 11

Q: A train travels 60 Km/h for 2 hours, then 90 Km/h for 1 hour. What's the total distance?
A: Step 1: First leg: 60 x 2 = 120 Km.
   Step 2: Second leg: 90 x 1 = 90 Km.
   Step 3: Total: 120 + 90 = 210 Km.
   Answer: 210 Km

Q: A rectangle has a perimeter of 30 cm. If the length is twice the width, what is the area?
A:'''

result = ask_open_ai(prompt)
display(Markdown(f"### Few-Shot CoT Result\n\n{result}"))

### Few-Shot CoT Result

Step 1: Let the width be W cm. Then the length is 2W cm.

Step 2: The formula for the perimeter of a rectangle is P = 2(length + width). So, we have:
P = 2(2W + W) = 30 cm.

Step 3: Simplifying the equation: 
2(3W) = 30.
This gives us 6W = 30.

Step 4: Dividing both sides by 6, we find W = 5 cm.

Step 5: Now, we can find the length: 
Length = 2W = 2 x 5 = 10 cm.

Step 6: The area of a rectangle is given by the formula A = length x width. 
So, A = 10 cm x 5 cm = 50 cm².

Answer: 50 cm²

---
## 6. CoT for Decision Making

In [12]:
decision = '''We need to choose a database for our new project.

Requirements:
- Flexible schema (data structure may change).
- High write throughtput (IoT sensor data).
- Horizontal scaling needed.
- Team has SQL experience.
- Budget is limited.

Options: PostgreSQL, MongoDB, DynamoDM, TimescaleDB

Analyse each option against our requirements step by step, then recommend one. 
'''
result = ask_open_ai(decision, max_resp_tokens=1500)
display(Markdown(result))

Let's analyze each option against your specified requirements:

### 1. PostgreSQL
- **Flexible Schema**: PostgreSQL supports JSONB, allowing for a degree of flexibility in schema design. However, it is fundamentally a relational database, which may require more effort for significant schema changes.
- **High Write Throughput**: PostgreSQL can handle a good amount of write operations, but it may not be as performant as some NoSQL solutions for extremely high write throughput scenarios.
- **Horizontal Scaling**: PostgreSQL traditionally scales vertically (adding more resources to a single server). It does have some options for horizontal scaling (like Citus), but this can add complexity and may not be as seamless as NoSQL solutions.
- **Team SQL Experience**: PostgreSQL uses SQL, making it easier for your team to adapt and utilize their existing skills.
- **Budget**: PostgreSQL is open-source and free, which is beneficial for a limited budget.

### 2. MongoDB
- **Flexible Schema**: MongoDB is a NoSQL database that allows for a flexible schema, making it easy to accommodate changing data structures.
- **High Write Throughput**: Designed for high write throughput, it can handle a large number of operations efficiently, which is ideal for IoT sensor data.
- **Horizontal Scaling**: MongoDB supports horizontal scaling natively through sharding, making it suitable for large-scale applications.
- **Team SQL Experience**: While MongoDB can be queried using a query language similar to SQL, it has a different paradigm that may require additional training for your team.
- **Budget**: MongoDB has a free tier but can become expensive with advanced features and managed services (like Atlas).

### 3. DynamoDB
- **Flexible Schema**: DynamoDB is a NoSQL database and offers flexibility with its schema design, allowing for dynamic attributes.
- **High Write Throughput**: It excels at high write throughput and is designed for scalability, making it well-suited for IoT applications.
- **Horizontal Scaling**: DynamoDB automatically scales horizontally without requiring much intervention, making it easy to handle growing workloads.
- **Team SQL Experience**: While DynamoDB has a SQL-like query language (PartiQL), it is fundamentally different from traditional SQL, which may require a learning curve for your team.
- **Budget**: DynamoDB is pay-per-use, which can be cost-effective if usage is managed carefully but may become expensive with heavy usage.

### 4. TimescaleDB
- **Flexible Schema**: TimescaleDB is built on PostgreSQL and retains a relational model, which can be somewhat flexible with time-series data but is less flexible than NoSQL databases.
- **High Write Throughput**: It is optimized for time-series data and can handle high write throughput efficiently, especially for IoT data.
- **Horizontal Scaling**: TimescaleDB supports horizontal scaling through multi-node configurations, but it is not as straightforward as some NoSQL databases.
- **Team SQL Experience**: Since it is based on PostgreSQL, your team can leverage their SQL experience effectively.
- **Budget**: TimescaleDB is open-source and has a community edition, making it budget-friendly.

### Recommendation
Considering your requirements, **MongoDB** emerges as the most suitable choice for your project. Here's why:

- It offers a highly flexible schema, which is essential given that your data structure may change.
- It provides high write throughput, which is crucial for handling IoT sensor data efficiently.
- It supports horizontal scaling out of the box, allowing you to grow as needed without significant architectural changes.
- While there may be a learning curve for your team due to its NoSQL nature, the flexibility and performance benefits of MongoDB could outweigh this.
- The potential costs can be managed with careful optimization and usage monitoring.

If your team is more inclined towards a SQL-based environment and you want to stay closer to traditional relational databases, **TimescaleDB** would be a strong second option, particularly if your IoT data has strong time-series characteristics. However, for maximum flexibility and scalability, MongoDB is recommended.

---
## 7. CoT Trigger Phrases

Different phrases work for different scenarios:

| Trigger | Best For |
|---------|----------|
| "Let's think step by step" | General reasoning |
| "Walk me through your reasoning" | Complex analyses |
| "Show your work" | Math problems |
| "Trace through the code" | Debugging |
| "Consider each option" | Decision making |
| "First, identify the key facts" | Reading comprehension |

In [14]:
triggers = [
    "Let's think step by step.",
    "Show your work.",
    "Walk me through your reasoning.",
]
 
problem = "What is 15% of 240?"
 
for trigger in triggers:
    result = ask_open_ai(f"{problem} {trigger}", max_resp_tokens=1500)
    display(Markdown(f"### Trigger: \"{trigger}\""))
    # print(f"Answer:  {result[:1000]}...\n")
    display(Markdown(f"### Answer:\n\n{result}"))
    print("-" * 71)

### Trigger: "Let's think step by step."

### Answer:

To find 15% of 240, follow these steps:

1. **Understand the percentage**: 15% means 15 out of every 100. This can also be expressed as a decimal: 15% = 0.15.

2. **Convert the percentage to a decimal**: 
   \[
   15\% = \frac{15}{100} = 0.15
   \]

3. **Multiply the decimal by the number**: 
   \[
   0.15 \times 240
   \]

4. **Calculate the multiplication**:
   - First, calculate \( 0.15 \times 240 \):
   \[
   0.15 \times 240 = 36
   \]

So, 15% of 240 is **36**.

-----------------------------------------------------------------------


### Trigger: "Show your work."

### Answer:

To find 15% of 240, you can follow these steps:

1. Convert the percentage to a decimal. 
   \[
   15\% = \frac{15}{100} = 0.15
   \]

2. Multiply the decimal by the number.
   \[
   0.15 \times 240
   \]

3. Perform the multiplication:
   \[
   0.15 \times 240 = 36
   \]

Therefore, 15% of 240 is **36**.

-----------------------------------------------------------------------


### Trigger: "Walk me through your reasoning."

### Answer:

To find 15% of 240, you can follow these steps:

1. **Convert the percentage to a decimal**: 
   - 15% as a decimal is 15 divided by 100, which equals 0.15.

2. **Multiply the decimal by the number**: 
   - Now, multiply 0.15 by 240:
   \[
   0.15 \times 240
   \]

3. **Calculate the multiplication**:
   - First, you can calculate:
   \[
   0.15 \times 240 = 36
   \]

Therefore, 15% of 240 is **36**.

-----------------------------------------------------------------------


---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **Zero-shot CoT** | Add "think step by step" — free accuracy boost |
| **Few-shot CoT** | Show reasoning examples for complex tasks |
| **Best for** | Math, logic, debugging, multi-step problems |
| **Not needed for** | Simple factual questions, translation |
| **Trade-off** | More tokens used (longer output) = higher cost |

---
**Next:** `04_pctf_framework.ipynb` — A systematic framework for writing effective prompts